In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats


In [ ]:
# Load raw CSV
df = pd.read_csv("tanzania.csv")

# Add country column
df["Country"] = "Tanzania"

# Convert YEAR + DOY to proper datetime
df["Date"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")

# Extract Month
df["Month"] = df["Date"].dt.month

print("Shape:", df.shape)
print("\nDate range:", df["Date"].min(), "to", df["Date"].max())
df.head()


In [ ]:
# Replace NASA sentinel value -999 with NaN BEFORE any statistics
df.replace(-999, np.nan, inplace=True)
print("Replaced -999 with NaN across entire DataFrame.")


In [ ]:
# Check for duplicate rows
n_dupes = df.duplicated().sum()
print(f"Duplicate rows found: {n_dupes}")
df.drop_duplicates(inplace=True)
print(f"Shape after dropping duplicates: {df.shape}")


In [ ]:
# Summary statistics on numeric columns
df.describe()


In [ ]:
# Missing value report
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_report = missing_report[missing_report['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print("Missing Value Report:")
print(missing_report)
print("\nColumns with >5% nulls:")
high_null = missing_report[missing_report['Missing %'] > 5]
print(high_null if not high_null.empty else "None — all columns are below 5% missing.")


In [ ]:
# Compute Z-scores for key weather variables
outlier_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']

z_scores = df[outlier_cols].apply(lambda col: stats.zscore(col, nan_policy='omit'))
outlier_mask = (z_scores.abs() > 3)
outlier_counts = outlier_mask.sum()

print("Outlier counts per column (|Z| > 3):")
print(outlier_counts)
print(f"\nTotal outlier rows (any column): {outlier_mask.any(axis=1).sum()}")


In [ ]:
# Handle missing values: forward-fill for weather variables
weather_vars = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE',
                'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']

# Drop rows where more than 30% of values are missing
threshold = int(0.7 * len(weather_vars))
before = len(df)
df.dropna(subset=weather_vars, thresh=threshold, inplace=True)
print(f"Rows dropped (>30% missing): {before - len(df)}")

# Forward-fill remaining missing values
df[weather_vars] = df[weather_vars].ffill()
print(f"Missing after forward-fill:\n{df[weather_vars].isna().sum()}")


In [ ]:
# Export cleaned data
import os
os.makedirs("data", exist_ok=True)
df.to_csv(f"data/tanzania_clean.csv", index=False)
print(f"Cleaned data saved to data/tanzania_clean.csv")
print(f"Final shape: {df.shape}")


In [ ]:
# Monthly average T2M
monthly_temp = df.groupby(['YEAR', 'Month'])['T2M'].mean().reset_index()
monthly_temp['Date'] = pd.to_datetime(monthly_temp[['YEAR', 'Month']].assign(Day=1))
monthly_temp = monthly_temp.sort_values('Date')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_temp['Date'], monthly_temp['T2M'], color='tomato', linewidth=1.5, alpha=0.9)

# Annotate warmest and coolest months
warmest = monthly_temp.loc[monthly_temp['T2M'].idxmax()]
coolest = monthly_temp.loc[monthly_temp['T2M'].idxmin()]

ax.annotate(f"Warmest\n{warmest['T2M']:.1f}°C",
            xy=(warmest['Date'], warmest['T2M']),
            xytext=(10, 10), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='darkred'),
            color='darkred', fontsize=9)

ax.annotate(f"Coolest\n{coolest['T2M']:.1f}°C",
            xy=(coolest['Date'], coolest['T2M']),
            xytext=(10, -20), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='steelblue'),
            color='steelblue', fontsize=9)

ax.set_title('Monthly Average Temperature (T2M) — 2015 to 2026', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
plt.show()


In [ ]:
# Monthly total precipitation bar chart
monthly_rain = df.groupby(['YEAR', 'Month'])['PRECTOTCORR'].sum().reset_index()
monthly_rain['Date'] = pd.to_datetime(monthly_rain[['YEAR', 'Month']].assign(Day=1))
monthly_rain = monthly_rain.sort_values('Date')

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['steelblue' if v < monthly_rain['PRECTOTCORR'].quantile(0.85) else 'navy'
          for v in monthly_rain['PRECTOTCORR']]
ax.bar(monthly_rain['Date'], monthly_rain['PRECTOTCORR'], width=25, color=colors, alpha=0.8)

# Annotate peak months
peak_idx = monthly_rain['PRECTOTCORR'].nlargest(3).index
for idx in peak_idx:
    row = monthly_rain.loc[idx]
    ax.annotate(f"{row['PRECTOTCORR']:.0f}mm",
                xy=(row['Date'], row['PRECTOTCORR']),
                xytext=(0, 5), textcoords='offset points',
                ha='center', fontsize=8, color='navy')

ax.set_title('Monthly Total Precipitation — 2015 to 2026', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Precipitation (mm)')

plt.show()


In [ ]:
# Correlation heatmap
numeric_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE',
                'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Correlation Heatmap — All Numeric Variables', fontsize=14, fontweight='bold')
plt.show()


In [ ]:
# Scatter: T2M vs RH2M
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['T2M'], df['RH2M'], alpha=0.3, color='steelblue', s=10)
axes[0].set_xlabel('T2M — Temperature (°C)')
axes[0].set_ylabel('RH2M — Relative Humidity (%)')
axes[0].set_title('Temperature vs Relative Humidity')

# Scatter: T2M_RANGE vs WS2M
axes[1].scatter(df['T2M_RANGE'], df['WS2M'], alpha=0.3, color='darkorange', s=10)
axes[1].set_xlabel('T2M_RANGE — Diurnal Temp Range (°C)')
axes[1].set_ylabel('WS2M — Wind Speed (m/s)')
axes[1].set_title('Diurnal Temp Range vs Wind Speed')

plt.suptitle('Scatter Plot Relationships', fontsize=14, fontweight='bold')
plt.show()


In [ ]:
# Histogram of PRECTOTCORR with log scale
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
axes[0].hist(df['PRECTOTCORR'].dropna(), bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Precipitation Distribution (Raw)')
axes[0].set_xlabel('PRECTOTCORR (mm)')
axes[0].set_ylabel('Frequency')

# Log scale
log_precip = np.log1p(df['PRECTOTCORR'].dropna())
axes[1].hist(log_precip, bins=60, color='navy', edgecolor='white', alpha=0.8)
axes[1].set_title('Precipitation Distribution (Log Scale)')
axes[1].set_xlabel('log(1 + PRECTOTCORR)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Precipitation Distribution Analysis', fontsize=14, fontweight='bold')
plt.show()


In [ ]:
# Bubble chart: T2M vs RH2M, bubble size = PRECTOTCORR
sample = df.sample(min(1000, len(df)), random_state=42)
bubble_size = (sample['PRECTOTCORR'].fillna(0) + 1) * 3

fig, ax = plt.subplots(figsize=(11, 7))
scatter = ax.scatter(
    sample['T2M'], sample['RH2M'],
    s=bubble_size, alpha=0.5,
    c=sample['PRECTOTCORR'], cmap='Blues', edgecolors='steelblue', linewidth=0.3
)
plt.colorbar(scatter, ax=ax, label='PRECTOTCORR (mm)')
ax.set_xlabel('T2M — Temperature (°C)', fontsize=12)
ax.set_ylabel('RH2M — Relative Humidity (%)', fontsize=12)
ax.set_title('Bubble Chart: Temperature vs Humidity\n(Bubble size & color = Precipitation)', fontsize=14, fontweight='bold')
plt.show()
